# Part 2 — When Tools Fail: retries, timeouts, and a failure taxonomy

*Agents from First Principles, Part 2.*

Part 1 gave every tool a **contract**: a typed JSON schema and a validator that rejects an unknown tool or a malformed argument *before* the call fires. That closes the door on calls that are wrong on paper. It does nothing about the call that is right on paper and still goes wrong at run time.

A schema-valid tool can still:
- **throw** -- a network blip, a 500, a rate limit, a bug in the tool;
- **time out** -- the dependency is up but slower than your deadline;
- **return garbage** -- an empty result, or a payload that passes every type check and still means "I found nothing."

RAG Part 19, and Part 1 here, assumed every tool **succeeds**: the loop took whatever a tool returned as ground truth and marched on. One throw and the whole run is a stack trace; one empty result silently poisons the answer. This part stops assuming success.

We wrap tool execution in a layer that **expects failure**. Three ideas, one per net-new mechanism:

1. **A failure taxonomy.** Not every failure is the same, and the right response depends on the *kind*. Five categories, one recovery policy each:

   | category | meaning | policy |
   |---|---|---|
   | `transient` | temporary; might work next time | **retry** (bounded, backoff) |
   | `permanent` | will never work; retrying wastes | **feed back** to the model |
   | `empty-result` | call worked, found nothing useful | **feed back** to the model |
   | `malformed` | bad arguments (Part 1's validator) | **feed back** to the model |
   | `unknown-tool` | not in the action space (Part 1) | **feed back** to the model |

   "Feed back" means: turn the failure into an **Observation** the controller reads and reasons about, instead of a crash. The loop survives a bad tool call the way it survives a good one.

2. **Retries with bounded exponential backoff and a timeout**, applied *only* to transient failures. Retrying a permanent error just burns time and money; the taxonomy is what tells the two apart.

3. **The first side-effecting tool: `process_refund()`.** A read-only tool can be retried for free. A tool that *moves money* cannot: if the refund posts and then the confirmation times out, a blind retry refunds the customer twice. The fix is an **idempotency guard** keyed by `order_id`, so a second attempt for the same order returns the first result instead of acting again. (A local in-memory guard here; Part 9 hardens it into durable idempotency keys that survive a process crash -- effectively-once across a replay. This part is the seed; Part 9 is the hardening.)

**Continuity**: same world as Part 1 (refund policy, the E-4042 error, the Acme to Globex chain) and the same deterministic-by-default runtime -- the controller is a readable rule policy offline, with `generate()` one env flag away.

> **Runs with no network, no API key, and no third-party dependency.** Set `OPENAI_API_KEY` to see the real LLM controller banner; the loop always falls through to the deterministic policy so output stays reproducible.

Companion script: `robust_execution.py`

## Setup

Two standard-library imports do the work: `os` lets us check for an API key without ever requiring one, and `re` powers the lexical retriever's tokenizer, the calculator's input guard, and the controller's order/amount matching. Nothing in the default path is imported from a third-party package, so every cell runs fully offline.

In [ ]:
import os
import re

print("ready -- no network, no API key, no third-party dependency required")

## Step 0 -- The world, carried verbatim from Part 1

The two corpora and the lexical retriever are exactly the ones Part 1 used, so nothing here is new -- it just keeps this notebook self-contained. `POLICY_KB` is the support corpus (refunds, the E-4042 error, shipping, warranty); `PRODUCTS` is the Acme-to-Globex acquisition chain. The retriever scores each chunk by content-word overlap, normalized by the geometric mean of the two token counts: a cosine-flavored score in [0, 1], deterministic and model-free so the demo's output is reproducible. The new material starts at Step 1.

In [ ]:
POLICY_KB = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Error E-4042 means the payment was declined by the bank; ask the customer to retry with a different card or contact their bank.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

PRODUCTS = [
    "Acme Corp was acquired by Globex in 2024.",
    "Globex now manufactures the wireless earbuds product line it inherited from Acme.",
    "Globex-branded wireless earbuds carry a 2-year limited warranty.",
    "The wireless earbuds deliver up to 8 hours of battery life, and up to 24 hours with the charging case.",
]

_TOKEN_RE = re.compile(r"[a-z0-9]+")
_STOPWORDS = {
    "what", "is", "are", "the", "of", "a", "an", "for", "on", "in", "to", "how",
    "do", "does", "and", "my", "i", "there", "with", "your", "our", "who", "by",
    "that", "made", "company", "s", "whats",
}


def _stem(tok):
    return tok[:-1] if len(tok) > 3 and tok.endswith("s") else tok


def _tokens(text):
    return [_stem(t) for t in _TOKEN_RE.findall(text.lower()) if t not in _STOPWORDS]


class _LexicalRetriever:
    """Deterministic, model-free stand-in for a dense retriever (RAG Part 6)."""

    def __init__(self, corpus):
        self.chunks = list(corpus)
        self._chunk_tokens = [set(_tokens(c)) for c in self.chunks]

    def _score(self, q_tokens, c_tokens):
        if not q_tokens or not c_tokens:
            return 0.0
        overlap = len(q_tokens & c_tokens)
        denom = (len(q_tokens) * len(c_tokens)) ** 0.5
        return overlap / denom

    def retrieve(self, query, k=1):
        q_tokens = set(_tokens(query))
        scored = [(self.chunks[i], self._score(q_tokens, self._chunk_tokens[i]))
                  for i in range(len(self.chunks))]
        scored.sort(key=lambda x: -x[1])
        return scored[:k]


_POLICY_STORE = _LexicalRetriever(POLICY_KB)
_PRODUCTS_STORE = _LexicalRetriever(PRODUCTS)

print(f"Policy KB: {len(POLICY_KB)} chunks. Products: {len(PRODUCTS)} chunks. Retriever ready (carried from Part 1).")

## Step 1 -- The taxonomy as exceptions

A tool signals *how* it failed by the **type** it raises, exactly as real code reads an HTTP status or a driver error code and decides "retry this" vs "give up on this." Two families:

- `TransientError` -- might clear on a retry (503, rate limit, timeout);
- `PermanentError` -- will not clear; a retry only wastes (404, bad request, auth).

A timeout is just a particular transient, so `ToolTimeout` *extends* `TransientError` -- the wrapper retries it for free by inheritance, no special case.

Below the exceptions are the five **categories** a call can resolve into (`OK` is the sixth, happy outcome) and the one-line `RECOVERY` policy for each. These strings are exactly what the trace prints, so the category a call lands in is visible in the output.

In [ ]:
class ToolError(Exception):
    """Base class for a tool failure the wrapper knows how to classify."""


class TransientError(ToolError):
    """Temporary: retrying with backoff may succeed."""


class ToolTimeout(TransientError):
    """The dependency was too slow for the deadline. A kind of transient."""


class PermanentError(ToolError):
    """Will never succeed for this call: do not retry; feed it back instead."""


# The five categories the wrapper resolves a call into; OK is the happy sixth.
OK = "ok"
TRANSIENT = "transient"
PERMANENT = "permanent"
EMPTY = "empty-result"
MALFORMED = "malformed"
UNKNOWN = "unknown-tool"

RECOVERY = {
    TRANSIENT: "RETRY (bounded, backoff)",
    PERMANENT: "FEED BACK to the model",
    EMPTY: "FEED BACK to the model",
    MALFORMED: "FEED BACK to the model",
    UNKNOWN: "FEED BACK to the model",
}

for cat, policy in RECOVERY.items():
    print(f"  {cat:<13} -> {policy}")

## Step 2a -- The read-only tools (carried) and the fault-injection tools

Two kinds of tool live here. The four **read-only tools** are unchanged from Part 1: `search_policy` and `search_products` wrap their index, `calculator` guards `eval` to digits and operators, and `finish` terminates the loop. Read-only tools are safe to retry for free -- calling `search_policy` twice changes nothing in the world. That is exactly *not* true of the side-effecting tool in the next cell, which is the whole reason this part exists.

Alongside them are four **fault-injection tools**, each standing in for a real tool whose dependency misbehaves in one specific way, so we can fire every branch of the taxonomy on demand:

- `flaky_lookup` -- fails twice (HTTP 503), then recovers: a **transient** that clears on retry;
- `slow_lookup` -- carries a `.latency` of 5.0s against a 2.0s deadline, so it **times out** every attempt;
- `broken_lookup` -- raises `PermanentError`: a 404 that retrying cannot fix;
- `empty_lookup` -- **succeeds** but returns `("", 0.0)`: an empty result, not an error.

The fault tools go in a separate registry (Step 3) used only by the fault demo, so the agent controller never sees them.

In [ ]:
def search_policy(query):
    """Retrieve the single best chunk from the POLICY index."""
    return _POLICY_STORE.retrieve(query, k=1)[0]      # (text, score)


def search_products(query):
    """Retrieve the single best chunk from the PRODUCTS index."""
    return _PRODUCTS_STORE.retrieve(query, k=1)[0]    # (text, score)


_CALC_RE = re.compile(r"^[\d\s+\-*/().%]+$")


def calculator(expression):
    """Evaluate simple arithmetic."""
    if not _CALC_RE.match(expression):
        return "calculator error: expression contains unsupported characters"
    try:
        return eval(expression, {"__builtins__": {}}, {})
    except Exception as exc:
        return f"calculator error: {type(exc).__name__}"


def finish(answer):
    """Terminate the loop with the final answer."""
    return answer


# --- Fault-injection tools: one controlled failure mode each. --------------
_FLAKY_STATE = {}


def flaky_lookup(query):
    """Transient: fails the first two calls (HTTP 503), then recovers."""
    _FLAKY_STATE["n"] = _FLAKY_STATE.get("n", 0) + 1
    if _FLAKY_STATE["n"] <= 2:
        raise TransientError("temporary upstream failure (HTTP 503)")
    return ("the dependency recovered and returned the data", 0.91)


def slow_lookup(query):
    """Too slow for the deadline on every attempt -> times out every time."""
    return ("(a result that never arrives in time)", 0.90)


slow_lookup.latency = 5.0      # simulated seconds; the wrapper's deadline is 2.0


def broken_lookup(query):
    """Permanent: the index does not exist. Retrying cannot fix it."""
    raise PermanentError("404: index 'archive' does not exist")


def empty_lookup(query):
    """The call SUCCEEDS but finds nothing useful. Not an error -- an empty result."""
    return ("", 0.0)


print("read-only tools: search_policy, search_products, calculator, finish")
print("fault tools: flaky_lookup, slow_lookup, broken_lookup, empty_lookup")

## Step 2b -- The first side-effecting tool: `process_refund()` with an idempotency guard

A refund **moves money**, so it cannot be retried blindly. The failure we simulate is the realistic one: the refund **posts** to the gateway, then the confirmation **times out** before we hear back. From the caller's side that looks identical to "it never happened" -- so a blind retry posts a *second* refund. The customer is charged back twice.

The fix is an **idempotency guard**. `_REFUND_LEDGER` maps `order_id` to the refund we recorded. The guard is the first line of `process_refund`: if the order is already in the ledger, return the recorded result *without* issuing a second refund. The crucial ordering: record the effect keyed by `order_id` **before** the point where we can fail, so a retry can recognize it. (In-memory here; Part 9 makes the record durable so it survives a process crash -- effectively-once across a replay.)

`process_refund_unsafe` is the identical twin with **no guard**, used once side-by-side in the demo to show what a blind retry does. It never appears in the agent loop.

In [ ]:
# _REFUND_LEDGER is the guard's memory: order_id -> the refund we recorded.
_REFUND_LEDGER = {}     # order_id -> {"amount": float}      (guarded path)
_REFUND_CALLS = {}      # order_id -> attempt count          (deterministic fault)


def process_refund(order_id, amount):
    """Issue a refund for an order. SIDE-EFFECTING and idempotent.

    The simulated failure is the realistic one: the refund POSTS to the gateway,
    then the confirmation times out before we hear back. A blind retry would
    refund twice. The guard prevents that: the effect is recorded keyed by
    order_id, so the retry sees it and skips re-acting.
    """
    if order_id in _REFUND_LEDGER:                    # IDEMPOTENCY GUARD
        prior = _REFUND_LEDGER[order_id]
        return (f"already refunded ${prior['amount']:.2f} to {order_id} "
                "(idempotent skip; no double-charge)")
    # Record the effect, keyed by the order id, BEFORE the point where we can
    # fail -- so a retry can recognize it. (Part 9 makes this record durable.)
    _REFUND_LEDGER[order_id] = {"amount": amount}
    _REFUND_CALLS[order_id] = _REFUND_CALLS.get(order_id, 0) + 1
    if _REFUND_CALLS[order_id] == 1:                  # first attempt: ack is lost
        raise TransientError("refund posted to the gateway, but the confirmation timed out")
    return f"refunded ${amount:.2f} to {order_id}"


# The UNSAFE twin: identical, but with NO guard. Used once, side by side, to show
# what a blind retry does to a side-effecting tool. Never used in the agent loop.
_UNSAFE_LEDGER = []     # appends one entry per successful POST (duplicates show up)
_UNSAFE_CALLS = {}


def process_refund_unsafe(order_id, amount):
    """A refund tool with no idempotency guard. Double-acts under retry."""
    _UNSAFE_CALLS[order_id] = _UNSAFE_CALLS.get(order_id, 0) + 1
    _UNSAFE_LEDGER.append({"order_id": order_id, "amount": amount})   # the effect
    if _UNSAFE_CALLS[order_id] == 1:
        raise TransientError("refund posted to the gateway, but the confirmation timed out")
    return f"refunded ${amount:.2f} to {order_id}"


print("side-effecting tools ready: process_refund (guarded), process_refund_unsafe (no guard)")

## Step 3 -- The registries and the validator (now returning a category)

`TOOL_SCHEMAS` is the agent's real **action space** -- the same Part 1 contract, now with `process_refund` added. `_FAULT_TOOLS` is a separate registry the standalone fault demo uses, so the controller never sees the misbehaving tools.

`validate_call` is Part 1's validator with one change: it now returns `(ok, error_message, category)`. An unknown tool is tagged `UNKNOWN`, a bad argument is tagged `MALFORMED`. These are the two pre-flight categories of the taxonomy: the validator catches them *before* the call fires, and the wrapper feeds them straight back without ever invoking the tool. The validator also takes the registry as an argument now, so the same function checks both the agent's tools and the fault tools.

In [ ]:
TOOL_SCHEMAS = {
    "search_policy": {
        "description": "search the support/policy index (refunds, errors, shipping, warranty)",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": search_policy,
    },
    "search_products": {
        "description": "search the products index (acquisitions, product warranties)",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": search_products,
    },
    "calculator": {
        "description": "evaluate simple arithmetic",
        "parameters": {"expression": {"type": "string", "required": True}},
        "fn": calculator,
    },
    "process_refund": {
        "description": "issue a refund for an order (side-effecting; idempotent by order id)",
        "parameters": {"order_id": {"type": "string", "required": True},
                       "amount": {"type": "number", "required": True}},
        "fn": process_refund,
    },
    "finish": {
        "description": "return the final answer and stop",
        "parameters": {"answer": {"type": "string", "required": True}},
        "fn": finish,
    },
}

_FAULT_TOOLS = {
    "flaky_lookup": {
        "description": "a lookup whose upstream fails transiently then recovers",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": flaky_lookup,
    },
    "slow_lookup": {
        "description": "a lookup slower than the deadline",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": slow_lookup,
    },
    "broken_lookup": {
        "description": "a lookup against an index that does not exist",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": broken_lookup,
    },
    "empty_lookup": {
        "description": "a lookup that succeeds but finds nothing",
        "parameters": {"query": {"type": "string", "required": True}},
        "fn": empty_lookup,
    },
    "process_refund_unsafe": {
        "description": "a refund tool with NO idempotency guard",
        "parameters": {"order_id": {"type": "string", "required": True},
                       "amount": {"type": "number", "required": True}},
        "fn": process_refund_unsafe,
    },
}

_PY_TYPE = {"string": str, "number": (int, float), "boolean": bool}


def validate_call(name, args, registry):
    """Return (ok, error_message, category). category is None when ok."""
    if name not in registry:
        return False, f"unknown tool '{name}' (not in the action space)", UNKNOWN
    schema = registry[name]["parameters"]
    for arg_name, spec in schema.items():
        if spec.get("required") and arg_name not in args:
            return False, f"{name} is missing required arg '{arg_name}'", MALFORMED
    for arg_name, value in args.items():
        if arg_name not in schema:
            return False, f"{name} got unexpected arg '{arg_name}'", MALFORMED
        expected = schema[arg_name]["type"]
        if not isinstance(value, _PY_TYPE[expected]):
            got = type(value).__name__
            return False, f"{name} arg '{arg_name}' must be {expected}, got {got}", MALFORMED
    return True, None, None


print("registries ready. Action space:", ", ".join(TOOL_SCHEMAS))

## Step 4 -- `execute_tool()`: the robustness layer

This is the heart of Part 2. One call goes through three stages:

1. **Pre-flight** -- run Part 1's validator. An unknown tool or malformed argument is *not* retried (a retry would fail identically); it is fed back at once as an Observation.
2. **Invoke under a deadline, retrying only transients.** A `PermanentError` returns immediately, fed back. A `TransientError` (which includes `ToolTimeout` by inheritance) is retried up to `MAX_RETRIES` times with **bounded exponential backoff**: the delay is `BASE_DELAY * BACKOFF**(attempt-1)`, i.e. 0.5s, 1.0s, 2.0s. When retries are exhausted, the error is fed back instead of crashing.
3. **Succeeded is not the same as found something.** A call that returns cleanly is still checked for emptiness (`_is_empty`): a blank payload or a retrieval score below `EMPTY_THRESHOLD` is the `empty-result` category, fed back so the model can try another approach.

The return value is always a `ToolOutcome` the loop can read -- `execute_tool` **never raises**. Offline, `_sleep` prints the planned backoff but does not actually wait (real code would `time.sleep` with jitter), and `_invoke_with_timeout` reads a simulated `.latency` so a timeout is deterministic rather than wall-clock.

In [ ]:
MAX_RETRIES = 2          # so up to 3 attempts total (1 try + 2 retries)
BASE_DELAY = 0.5         # seconds before the first retry
BACKOFF = 2.0            # each retry waits BACKOFF x longer: 0.5, 1.0, 2.0, ...
DEFAULT_TIMEOUT = 2.0    # per-call deadline, in (simulated) seconds
EMPTY_THRESHOLD = 0.1    # a retrieval score below this counts as "found nothing"


class ToolOutcome:
    """The resolved result of a tool call: which category it landed in, the
    observation text the loop should read, how many attempts it took, and whether
    it produced a usable result."""

    def __init__(self, category, observation, attempts):
        self.category = category
        self.observation = observation
        self.attempts = attempts
        self.usable = (category == OK)


def _backoff_delay(attempt):
    return BASE_DELAY * (BACKOFF ** (attempt - 1))    # 0.5, 1.0, 2.0, ...


def _sleep(seconds):
    """Offline: we PRINT the planned backoff but do not actually wait, so the
    demo is fast and reproducible. Real code would time.sleep(seconds) here, with
    a little random jitter so a fleet of clients does not retry in lockstep."""
    return None


def _invoke_with_timeout(fn, args, timeout):
    """Enforce a deadline. Real code would run fn in a thread/process and cancel
    it at the deadline (e.g. concurrent.futures .result(timeout=...)). Offline we
    read a simulated latency so a "timeout" is deterministic, not wall-clock."""
    latency = getattr(fn, "latency", 0.0)
    if latency > timeout:
        raise ToolTimeout(f"exceeded the {timeout:.1f}s deadline (simulated latency ~{latency:.1f}s)")
    return fn(**args)


def _unwrap(value):
    if isinstance(value, tuple):                      # a retriever's (text, score)
        return value[0], value[1]
    return value, None


def _obs_text(value):
    text, score = _unwrap(value)
    return f"{text} (score={score:.2f})" if score is not None else f"{text}"


def _short(value, n=72):
    t = _obs_text(value)
    return t if len(t) <= n else t[: n - 3] + "..."


def _is_empty(value):
    text, score = _unwrap(value)
    if score is not None:
        return (not str(text).strip()) or score < EMPTY_THRESHOLD
    return not str(text).strip()


def execute_tool(name, args, registry, timeout=DEFAULT_TIMEOUT, trace=None):
    """Run one tool call robustly. Returns a ToolOutcome; never raises."""
    def emit(line):
        if trace is not None:
            trace.append(line)

    # 1. Pre-flight: the Part 1 contract. Unknown tool / malformed args are not
    #    retried -- a retry would fail identically. Feed them back at once.
    ok, err, cat = validate_call(name, args, registry)
    if not ok:
        emit(f"    pre-flight -> {cat}: {err}")
        return ToolOutcome(cat, f"[{cat}] {err}", attempts=0)

    fn = registry[name]["fn"]
    # 2. Invoke under a deadline, retrying ONLY transient failures.
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            value = _invoke_with_timeout(fn, args, timeout)
        except PermanentError as exc:
            emit(f"    attempt {attempt} -> PermanentError: {exc}")
            emit(f"      {PERMANENT} -> {RECOVERY[PERMANENT]}")
            return ToolOutcome(PERMANENT, f"[{PERMANENT}] {exc}", attempt)
        except TransientError as exc:
            kind = type(exc).__name__
            if attempt <= MAX_RETRIES:
                delay = _backoff_delay(attempt)
                emit(f"    attempt {attempt} -> {kind}: {exc}")
                emit(f"      {TRANSIENT} -> retry {attempt}/{MAX_RETRIES} after {delay:.1f}s backoff")
                _sleep(delay)
                continue
            emit(f"    attempt {attempt} -> {kind}: {exc}")
            emit(f"      {TRANSIENT} -> retries exhausted ({MAX_RETRIES}); give up and feed back as an Observation")
            return ToolOutcome(TRANSIENT, f"[{TRANSIENT}, gave up after {attempt} attempts] {exc}", attempt)

        # 3. Succeeded without raising -- but "succeeded" is not "found something."
        if _is_empty(value):
            emit(f"    attempt {attempt} -> returned, but EMPTY")
            emit(f"      {EMPTY} -> {RECOVERY[EMPTY]}")
            return ToolOutcome(EMPTY, f"[{EMPTY}] the call succeeded but found nothing useful", attempt)
        emit(f"    attempt {attempt} -> OK: {_short(value)}")
        return ToolOutcome(OK, _obs_text(value), attempt)


print("execute_tool ready (deadline %.1fs, up to %d retries, backoff x%.0f)." %
      (DEFAULT_TIMEOUT, MAX_RETRIES, BACKOFF))

## Step 5 -- `generate()` and `build_prompt()`: the real LLM-driven path (reference shape only)

The same pattern as Part 1. In production the controller is an LLM: you hand it the goal, the running transcript, and `TOOL_SCHEMAS` as a rendered action space, and it emits the next Thought plus Action. The one addition for this part is in the system prompt: a tool result may now come back as an **error observation** (transient, permanent, empty-result, malformed, unknown-tool), and the model is told to read it and adapt -- a transient was already retried for it; a permanent or empty result means try another approach. The offline demo never calls `generate()`; the deterministic controller in Step 6 is the source of truth. Only `generate()` needs edits to go live.

In [ ]:
def _render_schemas():
    lines = []
    for name, s in TOOL_SCHEMAS.items():
        params = ", ".join(f"{p}: {spec['type']}" for p, spec in s["parameters"].items())
        lines.append(f"  {name}({params}) -- {s['description']}")
    return "\n".join(lines)


SYSTEM = """You are an augmented LLM that can call tools. Solve the goal step by
step. At each step output exactly:
  Thought: <reasoning>
  Action: <tool>(<json args>)
A tool result may come back as an error observation (transient, permanent,
empty-result, malformed, unknown-tool). Read it and adapt: a transient error was
already retried for you; a permanent or empty result means try another approach.
Action space:
{schemas}
Call finish(...) as soon as you can answer the goal."""


def build_prompt(goal, transcript):
    history = transcript if transcript else "(no steps yet)"
    return (SYSTEM.format(schemas=_render_schemas())
            + f"\n\nGoal: {goal}\n\nTranscript so far:\n{history}\n\nNext step:")


def generate(prompt):
    """REAL path: ask a hosted LLM for the next step. Unused offline."""
    from openai import OpenAI
    client = OpenAI()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content

# Anthropic / Claude alternative. Swap in for generate() above:
#
# def generate(prompt):
#     from anthropic import Anthropic
#     client = Anthropic()                            # reads ANTHROPIC_API_KEY
#     resp = client.messages.create(
#         model="claude-opus-4-8",                    # check current model names
#         max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text


if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM controller would drive "
          "generate(build_prompt(...)). Falling through to the deterministic policy "
          "so output stays reproducible.")
else:
    print("[controller] no OPENAI_API_KEY; using deterministic rule-based "
          "controller (offline default)")

## Step 6 -- The deterministic controller: the offline source of truth

Same device as Part 1: a readable rule policy keyed on the same cheap signals an LLM would weigh. This part's task is a refund, so the controller routes a refund goal in three steps: confirm the policy allows it, issue the refund, then finish. It pulls the order id and dollar amount straight out of the goal with two regexes. A real system swaps this body for one `generate()` call against `build_prompt()`; the loop in Step 7 is identical either way.

In [ ]:
_ORDER_RE = re.compile(r"ORD-\d+")
_AMOUNT_RE = re.compile(r"\$\s*(\d+(?:\.\d+)?)")


def controller(goal, steps):
    """Return the next (thought, tool, args) for the refund task."""
    n = len(steps)
    order = _ORDER_RE.search(goal)
    amount = _AMOUNT_RE.search(goal)
    order_id = order.group(0) if order else "ORD-0000"
    amt = float(amount.group(1)) if amount else 0.0

    if n == 0:
        return ("Before moving money, confirm refunds are allowed by policy.",
                "search_policy", {"query": "refund window 30 days"})
    if n == 1:
        return (f"Policy allows the refund; issue it for {order_id}.",
                "process_refund", {"order_id": order_id, "amount": amt})
    return ("The refund is recorded exactly once; finish.",
            "finish", {"answer": f"Refund of ${amt:.2f} for {order_id} is complete "
                                 "(processed once, despite a transient gateway timeout)."})


print("controller ready.")

## Step 7 -- The agent loop, every call through `execute_tool`

The loop is Part 1's reason/act/observe loop with one change that matters: every tool call goes through `execute_tool` instead of calling the function directly. A failure mid-run becomes an Observation the controller reads on the next step, not a stack trace that kills the run. `finish` is still validated (a malformed finish is caught), and the step budget is still the honest infinite-loop guard. `_reset_state` clears all the fault/refund state between demo sections so each one is independent of the order the others ran in.

In [ ]:
def _json_args(args):
    parts = [f'"{k}": ' + (f'"{v}"' if isinstance(v, str) else str(v)) for k, v in args.items()]
    return "{" + ", ".join(parts) + "}"


def run_agent(goal, registry, max_steps=6):
    steps = []                                        # (thought, tool, args, obs)
    for step in range(1, max_steps + 1):
        thought, tool, args = controller(goal, steps)
        print(f"  Step {step}")
        print(f"    Thought: {thought}")
        print(f"    Action: {tool}({_json_args(args)})")

        if tool == "finish":
            ok, err, _cat = validate_call(tool, args, registry)
            if not ok:
                print(f"      [invalid finish] {err}")
                steps.append((thought, tool, args, f"[invalid] {err}"))
                continue
            return args["answer"], step

        trace = []
        outcome = execute_tool(tool, args, registry, trace=trace)
        for line in trace:
            print(line)
        note = "" if outcome.attempts <= 1 else f"  (resolved after {outcome.attempts} attempts)"
        print(f"    Observation: {outcome.observation}{note}")
        steps.append((thought, tool, args, outcome.observation))

    return "Stopped: did not converge within the step budget.", max_steps


def _reset_state():
    """Clear all fault/refund state so each demo section is independent of the
    order the others ran in (keeps the output deterministic)."""
    _FLAKY_STATE.clear()
    _REFUND_LEDGER.clear()
    _REFUND_CALLS.clear()
    _UNSAFE_LEDGER.clear()
    _UNSAFE_CALLS.clear()


def _run_scenario(title, name, args, registry):
    print(f"\n  SCENARIO: {title}")
    print(f"    call: {name}({_json_args(args)})")
    trace = []
    outcome = execute_tool(name, args, registry, trace=trace)
    for line in trace:
        print(line)
    if outcome.category == OK:
        disp = f"RECOVERED, usable result after {outcome.attempts} attempt(s)"
    elif outcome.category == TRANSIENT:
        disp = f"GAVE UP gracefully after {outcome.attempts} attempts; error fed back as an Observation"
    else:
        disp = "fed back as an Observation; not retried"
    print(f"    disposition: {disp}  [category: {outcome.category}]")


print("run_agent and the scenario runner ready.")

## Demo -- the taxonomy reference, then everything that exercises it

Everything below runs fully offline, in four parts:

1. **The taxonomy reference** -- the five categories and their policies, printed.
2. **Fault injection** -- one scenario per branch, all driven through the *same* `execute_tool` wrapper, so you can watch each category resolve: `flaky_lookup` retries and recovers; `slow_lookup` times out, gives up gracefully, feeds back; `broken_lookup` is permanent and fed back on the first attempt; `empty_lookup` succeeds-but-empty; and the last two never reach the tool -- the validator catches the invented tool and the missing argument pre-flight.
3. **The side-effecting tool** -- the same transient (refund posts, ack times out) run once *without* a guard (the ledger shows two charges: a double refund) and once *with* (exactly one).
4. **The robust agent** -- the whole refund task in the loop. First the naive baseline (Part 1 style, no wrapper) dies on the first transient blip; then the robust loop confirms policy, issues the refund whose first attempt times out after posting, lets the retry hit the idempotency guard, and finishes with exactly one refund on the ledger.

In [ ]:
line = "=" * 72
print(line)
print("WHEN TOOLS FAIL  -  a failure taxonomy, retries, and an idempotent tool")
print(line)

if os.environ.get("OPENAI_API_KEY"):
    print("[controller] OPENAI_API_KEY set; the real LLM controller would drive "
          "generate(build_prompt(...)). Falling through to the deterministic policy "
          "so output stays reproducible.")
else:
    print("[controller] no OPENAI_API_KEY; using deterministic rule-based "
          "controller (offline default)")

# --- 1. The taxonomy reference. --------------------------------------------
print("\n" + "-" * 72)
print("THE FAILURE TAXONOMY: five kinds of failure, one recovery policy each.")
print("-" * 72)
print("  transient     temporary; might succeed if tried again   -> RETRY (bounded, backoff)")
print("  permanent     will never succeed; a retry only wastes    -> FEED BACK to the model")
print("  empty-result  the call worked but found nothing useful   -> FEED BACK to the model")
print("  malformed     arguments fail the schema (Part 1)         -> FEED BACK to the model")
print("  unknown-tool  not in the action space (Part 1)           -> FEED BACK to the model")
print('  "Feed back" = turn the failure into an Observation the controller reads, not a crash.')

# --- 2. Fault injection: one scenario per branch of the taxonomy. ----------
print("\n" + line)
print("FAULT INJECTION: drive each branch through the same execute_tool wrapper")
print(f"(deadline {DEFAULT_TIMEOUT:.1f}s, up to {MAX_RETRIES} retries, backoff x{BACKOFF:.0f}).")
print(line)
_reset_state()
_run_scenario("flaky-then-succeeds (transient that clears on retry)",
              "flaky_lookup", {"query": "today's order count"}, _FAULT_TOOLS)
_run_scenario("timeout (slower than the deadline, every attempt)",
              "slow_lookup", {"query": "today's order count"}, _FAULT_TOOLS)
_run_scenario("exception (permanent error: retrying cannot help)",
              "broken_lookup", {"query": "today's order count"}, _FAULT_TOOLS)
_run_scenario("empty-result (the call worked, found nothing)",
              "empty_lookup", {"query": "today's order count"}, _FAULT_TOOLS)
_run_scenario("unknown-tool (the model invented a tool)",
              "search_web", {"query": "today's order count"}, _FAULT_TOOLS)
_run_scenario("malformed (missing the required argument)",
              "flaky_lookup", {"q": "today's order count"}, _FAULT_TOOLS)

# --- 3. The side-effecting tool: why a retry needs an idempotency guard. ----
print("\n" + line)
print("THE SIDE-EFFECTING TOOL: a retry must NOT refund the customer twice.")
print("Same transient (refund posts, ack times out), once WITHOUT a guard, once WITH.")
print(line)

_reset_state()
print("\n  NO GUARD  process_refund_unsafe(ORD-5510, $80.00):")
trace = []
execute_tool("process_refund_unsafe", {"order_id": "ORD-5510", "amount": 80.0},
             _FAULT_TOOLS, trace=trace)
for ln in trace:
    print(ln)
print(f"    ledger: {len(_UNSAFE_LEDGER)} refunds posted for ORD-5510 -> "
      f"${sum(e['amount'] for e in _UNSAFE_LEDGER):.2f} charged back. DOUBLE REFUND.")

_reset_state()
print("\n  GUARDED   process_refund(ORD-5510, $80.00):")
trace = []
execute_tool("process_refund", {"order_id": "ORD-5510", "amount": 80.0},
             TOOL_SCHEMAS, trace=trace)
for ln in trace:
    print(ln)
print(f"    ledger: {len(_REFUND_LEDGER)} refund recorded for ORD-5510 -> "
      f"${_REFUND_LEDGER['ORD-5510']['amount']:.2f}. Exactly once.")
print("    The retry hit the guard and skipped re-acting. (Part 9 makes the guard "
      "durable so it")
print("    survives a process crash, not just this in-memory dict.)")

# --- 4. The whole thing in the loop. ---------------------------------------
print("\n" + line)
print("THE ROBUST AGENT: a transient failure mid-run no longer derails it.")
print("GOAL: Process the approved $49.99 refund for order ORD-7788.")
print(line)

_reset_state()
print("\n  Naive baseline (Part 1 style: call the tool directly, no wrapper):")
try:
    process_refund("ORD-7788", 49.99)
except ToolError as exc:
    print(f"    process_refund(...) raised {type(exc).__name__}: {exc}")
    print("    -> uncaught, the whole run dies on the first transient blip.")

_reset_state()
print("\n  Robust loop (every call through execute_tool):")
answer, steps_taken = run_agent("Process the approved $49.99 refund for order ORD-7788.",
                                TOOL_SCHEMAS)
print(f"  ANSWER: {answer}")
print(f"  ({steps_taken} steps; the refund's first attempt timed out after posting, the "
      "retry hit the")
print("   idempotency guard, and the run finished with exactly one refund on the ledger.)")

print("\n" + line)
print("Done. The layer between the loop and its tools:")
print("  - classify every failure (transient / permanent / empty / malformed / unknown)")
print("  - retry ONLY transients, bounded, with backoff; feed everything else back")
print("  - guard side-effecting tools so a retry never acts twice")
print("A schema-valid call can still fail; now a failure is an Observation, not a crash.")
print(line)

## Wrap-up

Part 1 made sure a tool call was *valid*. Part 2 makes sure that even a valid call cannot bring down the run. The robustness layer between the loop and its tools does three things:

- **classifies** every failure into the taxonomy (transient / permanent / empty / malformed / unknown);
- **retries only transients**, bounded, with exponential backoff and a deadline, and feeds everything else back as an Observation;
- **guards side-effecting tools** with an idempotency key so a retry never acts twice.

A schema-valid call can still fail at run time; now a failure is an Observation the controller reads, not a crash. The idempotency guard here is in-memory: it survives a retry within a single run, but not a process crash. Part 9 hardens this seed into durable, effectively-once idempotency keys. **Part 3 -- Planning and the tool DAG** is next: so far the controller has decided one step at a time; next it sketches the whole plan up front and runs the independent steps as a graph.